# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [4]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [5]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [6]:
# Keep every Asian country in the background, then draw the highlight on top.
asia = df.loc[df['Region'].eq('Asia')].copy()
highlight_country = 'China'
HIGHLIGHT_COLOR = '#D95F02'
CONTEXT_COLOR = '#EEEEEE'

highlight = asia.loc[asia['Country'].eq(highlight_country)].sort_values('Year')
others = asia.loc[asia['Country'].ne(highlight_country)]

fig = go.Figure()

# Context lines: light grey, thin, no legend entries
for country, country_data in others.groupby('Country'):
    country_data = country_data.sort_values('Year')
    fig.add_trace(go.Scatter(
        x=country_data['Year'],
        y=country_data['CO2_Mt'],
        mode='lines',
        name=country,
        line=dict(color=CONTEXT_COLOR, width=1.5),
        hovertemplate=f'{country}<br>%{{x}}: %{{y:,.0f}} Mt<extra></extra>',
        showlegend=False,
    ))

# Highlighted country drawn last so it sits on top of the grey lines
fig.add_trace(go.Scatter(
    x=highlight['Year'],
    y=highlight['CO2_Mt'],
    mode='lines',
    name=highlight_country,
    line=dict(color=HIGHLIGHT_COLOR, width=3),
    hovertemplate=f'{highlight_country}<br>%{{x}}: %{{y:,.0f}} Mt<extra></extra>',
    showlegend=False,
))

# Story numbers for the title, computed from the data rather than hard-coded
start_year, end_year = highlight['Year'].min(), highlight['Year'].max()
start_val = highlight.loc[highlight['Year'].eq(start_year), 'CO2_Mt'].iloc[0]
end_val = highlight.loc[highlight['Year'].eq(end_year), 'CO2_Mt'].iloc[0]
growth = end_val / start_val
runner_up = others.loc[others['Year'].eq(end_year)].nlargest(1, 'CO2_Mt').iloc[0]
lead = end_val / runner_up['CO2_Mt']

# End-point marker + direct label (no legend)
fig.add_trace(go.Scatter(
    x=[end_year], y=[end_val],
    mode='markers',
    marker=dict(color=HIGHLIGHT_COLOR, size=8),
    hoverinfo='skip',
    showlegend=False,
))
fig.add_annotation(
    x=end_year, y=end_val,
    text=f'<b>{highlight_country}</b><br>{end_val:,.0f} Mt',
    showarrow=False,
    xanchor='left', xshift=10, align='left',
    font=dict(color=HIGHLIGHT_COLOR, size=13),
)

# CO<sub>2</sub> is used instead of the subscript character to avoid encoding problems
fig.update_layout(
    title=dict(
        text=(
            f"{highlight_country}'s CO<sub>2</sub> emissions grew {growth:.1f}x "
            f"between {start_year} and {end_year}"
            f"<br><sup>By {end_year} it emitted {lead:.1f}x more than {runner_up['Country']}, "
            f"the next-largest Asian emitter in the dataset</sup>"
        ),
        x=0.02, xanchor='left',
        font=dict(size=18),
    ),
    xaxis_title=None,
    yaxis_title='Annual CO<sub>2</sub> emissions (Mt)',
    template='simple_white',
    showlegend=False,
    hovermode='closest',
    margin=dict(l=70, r=120, t=100, b=50),
)
fig.update_xaxes(showline=False, showgrid=False, zeroline=False, ticks='', dtick=5, tickangle=0)
fig.update_yaxes(showline=False, showgrid=True, gridcolor='#F5F5F5', zeroline=False, ticks='',
                 rangemode='tozero', tickformat=',')
fig.show()
